
# Steam Dataset — Exploratory Data Analysis

## Objective

This notebook profiles the raw Steam Bronze dataset in order to:

- understand dataset structure
- identify data quality issues
- support Silver-layer modeling decisions
- validate analytical assumptions
- answer preliminary business questions

Unlike the Silver notebook, this notebook focuses on:
- exploration
- profiling
- hypothesis validation
- anomaly detection

No transformations are persisted here.


## Imports

In [0]:

from pyspark.sql import functions as F


## Load Bronze Dataset

In [0]:

df_bronze = spark.table("steam.bronze.steam_games_bronze")



# 1. Dataset Structure

Objectives:
- inspect nested schema
- identify structs and arrays
- understand flattening requirements


In [0]:

df_bronze.printSchema()


root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:

display(df_bronze.limit(5))


data,id,ingestion_date
"List(1000540, List(Multi-player, Stats, Steam Achievements, Online PvP, PvP), 0, Hellride Games, 0, Casual, Free to Play, Indie, Strategy, https://cdn.akamai.steamstatic.com/steam/apps/1000540/header.jpg?t=1553210986, 0, English, Tactical Control, 12, 20,000 .. 50,000, List(false, false, true), 53, 0, Hellride Games, 2019/03/21, 0, Tactical Control: the 7-minute wargame. Simplified strategy with minimalist graphics and quick, intense games. Play short matches against random opponents. Fight tense battles of stealth, anticipation and surprise. Win with strategy, deception and cleverness, not with reflexes and clicking speed., List(null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 26, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 24, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 21, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 21, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null), game, )",1000540,2026-05-21T17:05:36.107Z
"List(1001500, List(Single-player, Steam Achievements, Steam Cloud), 0, Choice of Games, 0, Adventure, Indie, RPG, https://cdn.akamai.steamstatic.com/steam/apps/1001500/header.jpg?t=1547258371, 599, English, Chronicon Apocalyptica, 2, 0 .. 20,000, List(true, true, true), 6, 599, Choice of Games, 2019/01/11, 0, Battle Norse raiders, ghosts, and changelings to save medieval England! But beware, if the elves can capture the Book you hold, the world will end., List(null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 21, null, null, null, null, null, null, null, null, null


# 2. Dataset Dimensions

Objectives:
- estimate dataset scale
- validate ingestion completeness


In [0]:

display(
    spark.createDataFrame([
        ("rows", df_bronze.count()),
        ("columns", len(df_bronze.columns))
    ], ["metric", "value"])
)


metric,value
rows,55691
columns,3



# 3. Missing Values Profiling

Objectives:
- identify weak metadata fields
- estimate cleaning requirements
- detect problematic columns


In [0]:

null_summary = df_bronze.select([

    F.count(
        F.when(
            F.col(c).isNull(),
            c
        )
    ).alias(c)

    for c in df_bronze.columns
])

display(null_summary)


data,id,ingestion_date
0,0,0



# 4. Platform Availability Analysis

Business question:
- Are most games available on Windows, Mac or Linux?

Objectives:
- understand platform dominance
- validate platform boolean modeling


In [0]:

platform_summary = df_bronze.select(

    F.sum(
        F.col("data.platforms.windows").cast("int")
    ).alias("windows_games"),

    F.sum(
        F.col("data.platforms.mac").cast("int")
    ).alias("mac_games"),

    F.sum(
        F.col("data.platforms.linux").cast("int")
    ).alias("linux_games")
)

display(platform_summary)


windows_games,mac_games,linux_games
55676,12770,8458



# 5. Languages Exploration

Business question:
- What are the most represented languages?

Objectives:
- identify dominant languages
- detect malformed language metadata
- determine which language columns belong in Silver


In [0]:

df_languages = (

    df_bronze

    .select(
        F.explode(
            F.split(F.col("data.languages"), ",")
        ).alias("language")
    )

    .withColumn(
        "language",
        F.trim(F.col("language"))
    )

    .groupBy("language")
    .count()
    .orderBy(F.desc("count"))
)

display(df_languages)


language,count
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6600



# 6. Categories Exploration

Objectives:
- identify dominant gameplay features
- support category boolean modeling
- evaluate category cardinality


In [0]:

df_categories = (

    df_bronze

    .select(
        F.explode(F.col("data.categories")).alias("category")
    )

    .groupBy("category")
    .count()
    .orderBy(F.desc("count"))
)

display(df_categories)


category,count
Single-player,52025
Steam Achievements,27394
Steam Cloud,14235
Full controller support,11879
Multi-player,11455
Steam Trading Cards,9208
Partial Controller Support,7867
PvP,7070
Co-op,5616
Steam Leaderboards,5509



# 7. Genre Exploration

Business questions:
- What are the most represented genres?
- Do publishers specialize in certain genres?
- Which genres appear most lucrative?

Objectives:
- evaluate genre cardinality
- validate exploded genre table modeling
- inspect malformed genre values


In [0]:

df_genres = (

    df_bronze

    .select(
        F.explode(
            F.split(F.col("data.genre"), ",")
        ).alias("genre")
    )

    .withColumn(
        "genre",
        F.trim(F.col("genre"))
    )

    .groupBy("genre")
    .count()
    .orderBy(F.desc("count"))
)

display(df_genres)


genre,count
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


In [0]:

display(
    df_bronze.filter(
        F.col("data.genre").isNull()
    )
)


data,id,ingestion_date



# 8. Pricing & Discount Exploration

Business questions:
- How are prices distributed?
- Are there many discounted games?

Objectives:
- inspect pricing units
- detect outliers
- validate casting strategy


In [0]:

display(
    df_bronze.select("data.price")
    .distinct()
    .orderBy("price")
)


price
0
100
1000
1004
1019
1025
103
1039
104
1049


In [0]:

display(
    df_bronze.select("data.initialprice")
    .distinct()
    .orderBy("initialprice")
)


initialprice
0
100
1000
1025
1029
1099
1149
1190
1199
11999


In [0]:

display(
    df_bronze.select("data.discount")
    .distinct()
    .orderBy("discount")
)


discount
0
10
12
15
18
19
20
21
22
24



# 9. Required Age Exploration

Business question:
- Are many games restricted for minors?

Objectives:
- identify malformed age values
- define harmonization rules


In [0]:

display(
    df_bronze.select("data.required_age")
    .distinct()
    .orderBy("required_age")
)


required_age
0
10
12
13
14
15
16
17
18
180


In [0]:

display(
    df_bronze.filter(
        F.col("data.required_age").contains("+")
    )
)


data,id,ingestion_date
"List(1817140, List(Single-player), 0, Kennedy Demir, Jonathan Demir, 0, Action, Adventure, https://cdn.akamai.steamstatic.com/steam/apps/1817140/header.jpg?t=1663752045, 299, English, Ending Way, 0, 0 .. 20,000, List(false, false, true), 3, 299, Kennedy Demir, Jonathan Demir, 2021/12/27, 7+, [Chapter 3] chapter 3 is now here waiting for you. With horrible creatures and things you can do to get out. But ..., List(null, null, null, null, null, null, null, 31, null, 33, null, null, null, null, null, null, null, 66, null, null, null, 36, null, 60, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 24, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 19, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 21, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 25, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null), game, )",1817140,2026-05-21T17:05:36.107Z
"List(2120100, List(Single-player), 0, Top Fun Gaming Inc., 0, Action, Adventure, https://cdn.akamai.steamstatic.com/steam/apps/2120100/header.jpg?t=1667998199, 499, English, Top Fun 10 VR, 1, 0 .. 20,000, List(false, false, true), 0, 499, Top Fun Gaming Inc., 2022/09/10, 21+, A virtual reality game where you explore towns and forests where you have to fight for your life to find your princess and save her from the enemy. Fight zombies and magical creatures an a open world. You have all kinds of fighting abilities including hand to hand combat., List(null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 69, 43, null, null, 40, null, 21, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null


# 10. Owners Exploration

Business questions:
- Which games appear most successful?
- Which genres seem most lucrative?

Objectives:
- inspect owners range format
- validate midpoint engineering strategy


In [0]:

display(
    df_bronze.select("data.owners")
    .distinct()
    .orderBy("owners")
)


owners
"0 .. 20,000"
"1,000,000 .. 2,000,000"
"10,000,000 .. 20,000,000"
"100,000 .. 200,000"
"2,000,000 .. 5,000,000"
"20,000 .. 50,000"
"20,000,000 .. 50,000,000"
"200,000 .. 500,000"
"200,000,000 .. 500,000,000"
"5,000,000 .. 10,000,000"



# 11. Release Date Exploration

Business question:
- Are there years with more releases?

Objectives:
- identify malformed dates
- inspect format heterogeneity
- support release date standardization


In [0]:

display(
    df_bronze.select("data.release_date")
    .distinct()
)


release_date
2019/03/21
2019/01/11
2019/04/15
2019/01/26
2019/02/27
2019/02/15
2020/04/17
2019/04/3
2019/03/13
2020/03/7


In [0]:

display(
    df_bronze.filter(
        F.col("data.release_date").isNull()
    )
)


data,id,ingestion_date



# 12. Review Metrics Exploration

Business question:
- What are the best rated games?

Objectives:
- inspect review distributions
- validate positive_ratio KPI engineering


In [0]:

display(
    df_bronze.select(
        "data.positive",
        "data.negative",
        "data.ccu"
    )
)


positive,negative,ccu
53,12,0
6,2,0
10,6,0
20,9,0
30,17,0
26,21,0
8,3,0
10,4,0
4,0,0
9,9,0


In [0]:
display(

    df_bronze.select(

        F.when(
            (
                F.col("data.positive") +
                F.col("data.negative")
            ) > 0,

            F.round(
                F.col("data.positive") /
                (
                    F.col("data.positive") +
                    F.col("data.negative")
                ),
                4
            )
        )

        .otherwise(None)

        .alias("positive_ratio_preview")
    )
)

positive_ratio_preview
0.8154
0.75
0.625
0.6897
0.6383
0.5532
0.7273
0.7143
1.0
0.5



# 13. Publisher & Developer Exploration

Business question:
- Which publisher released the most games?

Objectives:
- inspect missing metadata
- identify dominant publishers
- inspect publisher/developer consistency
- evaluate ecosystem concentration


In [0]:

display(

    df_bronze.select(

        F.count(
            F.when(
                F.col("data.publisher").isNull() |
                (F.trim(F.col("data.publisher")) == ""),
                1
            )
        ).alias("missing_publishers"),

        F.count(
            F.when(
                F.col("data.developer").isNull() |
                (F.trim(F.col("data.developer")) == ""),
                1
            )
        ).alias("missing_developers")
    )
)


missing_publishers,missing_developers
134,127


In [0]:

top_publishers = (

    df_bronze

    .groupBy("data.publisher")
    .count()
    .orderBy(F.desc("count"))
)

display(top_publishers.limit(20))


publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
,132
Ubisoft,127


In [0]:

top_developers = (

    df_bronze

    .groupBy("data.developer")
    .count()
    .orderBy(F.desc("count"))
)

display(top_developers.limit(20))


developer,count
Choice of Games,140
,127
Creobit,122
Laush Dmitriy Sergeevich,108
Sokpop Collective,98
"KOEI TECMO GAMES CO., LTD.",90
Reforged Group,89
Boogygames Studios,80
Hosted Games,79
Elephant Games,75


In [0]:

publisher_developer_relation = (

    df_bronze

    .select(

        F.when(
            F.col("data.publisher") == F.col("data.developer"),
            "same"
        )

        .otherwise("different")

        .alias("publisher_developer_relation")
    )

    .groupBy("publisher_developer_relation")
    .count()
)

display(publisher_developer_relation)


publisher_developer_relation,count
same,37448
different,18243



# 14. EDA Synthesis

Main findings:

- Steam metadata contains nested arrays and structs
- platform modeling is well-suited for boolean features
- languages and categories require profiling before Silver modeling
- genre analysis benefits from exploded analytical modeling
- release dates contain heterogeneous formats
- pricing fields require normalization
- owners ranges require midpoint engineering
- publisher ecosystem appears highly long-tail distributed
- several metadata fields require null harmonization
